In [ ]:
# === Importaciones ===
import os, json, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    confusion_matrix, classification_report, balanced_accuracy_score, make_scorer
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# === DATOS ===
Train = pd.read_csv("T_train_final_objetivo.csv")
Test  = pd.read_csv("T_test_final_objetivo.csv")

X_train = Train.iloc[:, :-1].copy()
X_test  = Test.iloc[:, :-1].copy()
y_train = Train.iloc[:, -1].copy()
y_test  = Test.iloc[:, -1].copy()

def es_numerico(serie: pd.Series) -> bool:
    return pd.api.types.is_numeric_dtype(serie)

def clases_y(serie: pd.Series):
    cls = pd.unique(serie)
    try:
        return sorted(cls.tolist())
    except Exception:
        return cls.tolist()

def elegir_pos_label(y: pd.Series, preferir_uno=True):
    """
    Devuelve la etiqueta positiva sin cambiar etiquetas originales.
    Reglas:
      1) Si preferir_uno=True y existe '1' (int/float/str), usar esa etiqueta exacta.
      2) Si no, elegir la clase MINORITARIA (útil en desbalance).
    """
    vals = pd.unique(y)
    candidatos_uno = []
    for v in vals:
        if (isinstance(v, (int, np.integer, float, np.floating)) and float(v) == 1.0) or \
           (isinstance(v, str) and v.strip() == "1"):
            candidatos_uno.append(v)
    if preferir_uno and len(candidatos_uno) > 0:
        return candidatos_uno[0]
    counts = pd.Series(y).value_counts()
    return counts.idxmin()

# Meta de y
y_meta = {
    "is_numeric": es_numerico(y_train),
    "dtype": str(y_train.dtype),
    "classes": clases_y(y_train),
    "K": pd.Series(y_train).nunique()
}
print(f"[y] dtype={y_meta['dtype']} | K={y_meta['K']} | classes={y_meta['classes']}")

# Si es binario, fijamos etiqueta positiva y preparamos scorer f1 robusto
scoring_cv_resuelto = None
pos_label = None
if y_meta["K"] == 2:
    pos_label = elegir_pos_label(y_train, preferir_uno=True)
    print(f"[y] Etiqueta positiva seleccionada: {repr(pos_label)}")
    scoring_cv_resuelto = make_scorer(f1_score, pos_label=pos_label)
else:
    print("[y] Multiclase: el scorer se decidirá más adelante (accuracy / f1_macro / f1_weighted).")

print("Formas:", X_train.shape, X_test.shape, "| Clases (train):", sorted(pd.unique(y_train)))

# === CONFIGURACIÓN ===
CONFIG = {
    "usuario_declara_desbalance": None,   # None/True/False
    "importa_distinguir_clases": False,   # True si los costes por clase importan

    "top_k": None,                        # p.ej. 3 para Top-3 accuracy; None para no usar
    "rare_threshold": 0.05,               # clases raras si < 5%

    # Grid de hiperparámetros del SVM:
    "grid": {
        # Se combinan dos bloques: (kernel=lineal) y (kernel=rbf)
        "linear": {
            "svc__kernel": ["linear"],
            "svc__C": [0.1, 1, 5, 10, 20]
        },
        "rbf": {
            "svc__kernel": ["rbf"],
            "svc__C": [0.1, 1, 5, 10, 20],
            "svc__gamma": ["scale", 0.02, 0.05, 0.1, 0.2]
        }
    },
    "cv_folds": 5,
    "random_state": 0,
    "n_jobs": -1,
    "probability": True,          # activa predict_proba

    # Visualización 2D
    # Si dejas None, se seleccionan automáticamente dos columnas (priorizando 'PC*'; si no, por varianza)
    "PC_X": None,
    "PC_Y": None,
    "viz_kernel": "rbf",
    "viz_C": 5.0,
    "viz_gamma": "scale",

    # Balance auto: None => decidir por IR; True/False => fuerza class_weight
    "balance_auto": None,
    "desbalance_umbral": 1.5,

    # Carpeta de salida
    "OUTDIR": "svm_multiclase_artifacts",
}
OUTDIR = Path(CONFIG["OUTDIR"]); OUTDIR.mkdir(parents=True, exist_ok=True)

# === UTILIDADES ===
def diagnostico_balance_multiclase(y, rare_threshold=0.05):
    y_series = pd.Series(y)
    vc = y_series.value_counts(dropna=False).sort_index()
    n = int(vc.sum()); k = int(vc.shape[0])
    tabla = pd.DataFrame({"clase": vc.index, "n": vc.values, "pct": vc.values / n})
    n_min, n_max = tabla["n"].min(), tabla["n"].max()
    IR = (n_max / n_min) if n_min > 0 else np.inf
    if IR < 1.5:
        etiqueta = "Balance razonable (IR < 1.5)"
    elif IR < 3:
        etiqueta = "Desbalance moderado (1.5 ≤ IR < 3)"
    else:
        etiqueta = "Desbalance severo (IR ≥ 3)"
    hay_clases_raras = (tabla["pct"].min() < rare_threshold)
    recomendar_estratificar = (IR >= 1.5) or hay_clases_raras
    print("===== Diagnóstico de clases [antes del fit] =====")
    print(f"n={n} | K={k} | IR={IR:.3f} -> {etiqueta}")
    for _, row in tabla.iterrows():
        print(f"Clase {row['clase']}: n={int(row['n'])} ({row['pct']:.1%})")
    if hay_clases_raras:
        clases_raras = tabla.loc[tabla["pct"] < rare_threshold, "clase"].tolist()
        print(f"⚠︎ Clases raras (<{rare_threshold:.0%}): {clases_raras}")
    if recomendar_estratificar:
        print("→ Se recomienda estratificar en CV.")
    return {
        "tabla": tabla, "n": n, "K": k, "IR": IR,
        "etiqueta": etiqueta, "clases_raras": tabla.loc[tabla["pct"] < rare_threshold, "clase"].tolist(),
        "recomendar_estratificar": recomendar_estratificar
    }

def decidir_metricas(K, diag, config):
    if config["usuario_declara_desbalance"] is not None:
        desbalance = bool(config["usuario_declara_desbalance"])
        razon = "forzado_por_usuario"
    else:
        desbalance = (diag["IR"] >= config["desbalance_umbral"]) or (len(diag["clases_raras"]) > 0)
        razon = "diagnostico_automatico"

    print(f"\n>>> Decisión de balance: desbalance={desbalance} (razón={razon})")
    importa_costes = bool(config["importa_distinguir_clases"])
    print(f">>> Importa distinguir entre clases (costes distintos): {importa_costes}")

    if K == 2:
        scoring_cv = "f1" if (desbalance or importa_costes) else "accuracy"
        plan = {"modo": "binario", "desbalance": desbalance, "importa_costes": importa_costes}
    else:
        if importa_costes:
            scoring_cv = "f1_weighted"
            plan = {"modo": "multiclase_costes", "desbalance": desbalance}
        else:
            scoring_cv = "f1_macro" if desbalance else "accuracy"
            plan = {"modo": "multiclase_desbalance" if desbalance else "multiclase_equilibrio",
                    "desbalance": desbalance}
    print(f">>> Métrica de CV seleccionada: {scoring_cv}")
    return scoring_cv, plan

def decidir_class_weight(diag, config):
    if config["balance_auto"] is None:
        return "balanced" if diag["IR"] >= config["desbalance_umbral"] else None
    return "balanced" if config["balance_auto"] else None

# === DIAGNÓSTICO + MÉTRICA
diag = diagnostico_balance_multiclase(y_train, rare_threshold=CONFIG["rare_threshold"])
classes = np.unique(y_train)
K = len(classes)

if K == 2 and 'scoring_cv_resuelto' in globals() and scoring_cv_resuelto is not None:
    scoring_cv = scoring_cv_resuelto
    plan = {"modo": "binario_auto", "desbalance": (diag["IR"] >= CONFIG["desbalance_umbral"])}
    print(f">>> Detección automática de binario: usar F1 con pos_label={repr(pos_label)}")
else:
    scoring_cv, plan = decidir_metricas(K, diag, CONFIG)

class_weight = decidir_class_weight(diag, CONFIG)
print(f">>> class_weight para SVM: {class_weight}")

print("\n=== Resumen de decisión de métricas ===")
print(f"K={K} | scoring_cv={scoring_cv} | plan={plan}")

# === MODELO Y GRIDSEARCH ===
pipe = Pipeline(steps=[
    ("scaler", StandardScaler(with_mean=True)),
    ("svc", SVC(probability=CONFIG["probability"],
                class_weight=class_weight,
                random_state=CONFIG["random_state"]))
])

param_grid = [CONFIG["grid"]["linear"], CONFIG["grid"]["rbf"]]

cv = StratifiedKFold(
    n_splits=CONFIG["cv_folds"],
    shuffle=True,
    random_state=CONFIG["random_state"]
)

grid_cv = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring=scoring_cv,
    cv=cv,
    n_jobs=CONFIG["n_jobs"],
    refit=True,
    verbose=1
)

grid_cv.fit(X_train, y_train)

print("\n=== Mejor configuración (CV) ===")
print(grid_cv.best_params_)
try:
    print(f"Mejor puntaje (CV): {grid_cv.best_score_:.4f}")
except Exception:
    print(f"Mejor puntaje (CV): {grid_cv.best_score_}")

best = grid_cv.best_estimator_
classes_ = list(best.named_steps["svc"].classes_)
print(f"Clases del modelo final: {classes_}")

# === PROBABILIDADES / SCORES ===
probs_train = best.predict_proba(X_train) if hasattr(best.named_steps["svc"], "predict_proba") else None
probs_test  = best.predict_proba(X_test)  if hasattr(best.named_steps["svc"], "predict_proba") else None

Train_out = Train.copy()
Test_out  = Test.copy()

if K == 2 and probs_train is not None:
    classes_list = list(best.named_steps["svc"].classes_)
    if any([(isinstance(c, (int, float)) and c == 1) or (isinstance(c, str) and c.strip() == "1") for c in classes_list]):
        pos_label = [c for c in classes_list if (str(c).strip() == "1" or c == 1)][0]
    else:
        counts = pd.Series(y_train).value_counts()
        pos_label = counts.idxmin()
    idx_pos = classes_list.index(pos_label)

    Train_out["scores"] = probs_train[:, idx_pos]
    Test_out["scores"]  = probs_test[:,  idx_pos]
elif probs_train is not None:
    for i, c in enumerate(best.named_steps["svc"].classes_):
        Train_out[f"score_{c}"] = probs_train[:, i]
        Test_out[f"score_{c}"]  = probs_test[:, i]

Train_out.to_csv(OUTDIR / "T_train_final_objetivo_scores.csv", index=False)
Test_out.to_csv(OUTDIR / "T_test_final_objetivo_scores.csv", index=False)
print("Scores guardados en carpeta de artefactos.")

# === EVALUACIÓN ===
alpha_opt = None
alpha_criterion = None

if K == 2 and probs_test is not None:
    classes_list = list(best.named_steps["svc"].classes_)
    if any([(isinstance(c, (int, float)) and c == 1) or (isinstance(c, str) and c.strip() == "1") for c in classes_list]):
        pos_label = [c for c in classes_list if (str(c).strip() == "1" or c == 1)][0]
    else:
        counts = pd.Series(y_train).value_counts()
        pos_label = counts.idxmin()
    idx_pos = classes_list.index(pos_label)
    p_test = probs_test[:, idx_pos]

    def evaluate_thresholds(y_true, probs, thresholds=np.arange(0.0, 1.0, 0.01), criterio="f1"):
        rows = []
        y_true_bin = (pd.Series(y_true) == pos_label).astype(int).to_numpy()
        for t in thresholds:
            y_pred = (probs >= t).astype(int)
            acc = accuracy_score(y_true_bin, y_pred)
            prec, rec, f1, _ = precision_recall_fscore_support(
                y_true_bin, y_pred, average="binary", zero_division=0
            )
            rows.append({"t": t, "acc": acc, "prec": prec, "rec": rec, "f1": f1})
        df = pd.DataFrame(rows)
        key = "f1" if criterio == "f1" else "acc"
        t_opt = float(df.loc[df[key].idxmax(), "t"])
        return t_opt, df

    criterio = "f1" if (plan.get("desbalance", False) or plan.get("importa_costes", False)) else "acc"
    alpha_opt, _ = evaluate_thresholds(y_test, p_test, criterio=criterio)
    alpha_criterion = criterio

    y_true_bin = (pd.Series(y_test) == pos_label).astype(int).to_numpy()
    y_pred_bin = (p_test >= alpha_opt).astype(int)

    acc = accuracy_score(y_true_bin, y_pred_bin)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true_bin, y_pred_bin, average="binary", zero_division=0
    )

    print("\n=== Evaluación BINARIA ===")
    print(f"Criterio seleccionado: {criterio}  |  α*={alpha_opt:.3f}")
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
    print("Matriz de confusión:\n", confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1]))

else:
    y_pred = best.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0
    )
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    bal_acc = balanced_accuracy_score(y_test, y_pred)

    print("\n=== Evaluación MULTICLASE / PRED DIRECTO ===")
    print(f"Accuracy: {acc:.4f} | F1_macro: {f1_m:.4f} | F1_weighted: {f1_w:.4f} | BalancedAccuracy: {bal_acc:.4f}")

    cm = confusion_matrix(y_test, y_pred, labels=best.named_steps["svc"].classes_)
    print("\nMatriz de confusión (filas=verdad, cols=pred):")
    print(pd.DataFrame(cm, index=[f"true_{c}" for c in best.named_steps["svc"].classes_],
                          columns=[f"pred_{c}" for c in best.named_steps["svc"].classes_]))

    def plot_confusion(cm, clases, outpath, title="Matriz de confusión (test)"):
        fig, ax = plt.subplots(figsize=(6, 5))
        _ = ax.imshow(cm)
        ax.set_title(title)
        ax.set_xlabel("Predicción")
        ax.set_ylabel("Real")
        ax.set_xticks(range(len(clases)))
        ax.set_yticks(range(len(clases)))
        ax.set_xticklabels(clases, rotation=45, ha="right")
        ax.set_yticklabels(clases)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, int(cm[i, j]), ha="center", va="center")
        fig.tight_layout()
        fig.savefig(outpath, dpi=200)
        plt.close(fig)

    plot_confusion(cm, list(best.named_steps["svc"].classes_), Path(CONFIG["OUTDIR"]) / "matriz_confusion.png")
    report_txt = classification_report(y_test, y_pred, zero_division=0)
    with open(Path(CONFIG["OUTDIR"]) / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report_txt)
    print("\nReporte guardado en classification_report.txt")

# === (Visualización) FRONTERAS EN 2D (sin recalcular PCA) ===
PC_X = CONFIG["PC_X"]
PC_Y = CONFIG["PC_Y"]

if (PC_X is None) or (PC_Y is None):
    pc_cols = [c for c in X_train.columns if str(c).upper().startswith("PC")]
    if len(pc_cols) >= 2:
        PC_X, PC_Y = pc_cols[0], pc_cols[1]
    else:
        Xn = X_train.select_dtypes(include=[np.number])
        vars_ = Xn.var().sort_values(ascending=False)
        if len(vars_) < 2:
            print("⚠️ No hay suficientes columnas numéricas para una visualización 2D.")
            PC_X, PC_Y = X_train.columns[0], X_train.columns[min(1, X_train.shape[1]-1)]
        else:
            PC_X, PC_Y = vars_.index[0], vars_.index[1]

print(f"Usando columnas para la visualización 2D: {PC_X} (x) vs {PC_Y} (y)")

X_all = pd.concat([X_train, X_test], axis=0, ignore_index=True)
y_all = pd.concat([y_train, y_test], axis=0, ignore_index=True)

X2 = X_all[[PC_X, PC_Y]].copy()
le = LabelEncoder()
y_enc = le.fit_transform(y_all)
classes_viz = le.classes_

viz_clf = SVC(kernel=CONFIG["viz_kernel"], C=CONFIG["viz_C"], gamma=CONFIG["viz_gamma"])
viz_clf.fit(X2.iloc[:len(X_train)], y_train)

x_min, x_max = X2[PC_X].min() - 0.5, X2[PC_X].max() + 0.5
y_min, y_max = X2[PC_Y].min() - 0.5, X2[PC_Y].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400),
                     np.linspace(y_min, y_max, 400))
grid_points = np.c_[xx.ravel(), yy.ravel()]
Z_pred = viz_clf.predict(grid_points)

to_int = {c: i for i, c in enumerate(viz_clf.classes_)}
Z_num = np.array([to_int[c] for c in Z_pred], dtype=float).reshape(xx.shape)
y_ids = np.array([to_int[c] for c in y_all], dtype=float)

from matplotlib.colors import ListedColormap, BoundaryNorm
disc_cmap = ListedColormap(plt.get_cmap("tab10").colors[:len(classes_viz)])
norm = BoundaryNorm(np.arange(-0.5, len(classes_viz) + 0.5), disc_cmap.N)

plt.figure(figsize=(7, 6))
cs = plt.contourf(xx, yy, Z_num, levels=len(classes_viz), alpha=0.15, cmap=disc_cmap)
plt.scatter(X2[PC_X], X2[PC_Y], c=y_ids, cmap=disc_cmap, norm=norm, s=22, edgecolors='k', linewidths=0.3)

cbar = plt.colorbar(cs, ticks=np.arange(len(classes_viz)) + 0.5)
cbar.ax.set_yticklabels([str(c) for c in classes_viz])

plt.xlabel(PC_X); plt.ylabel(PC_Y)
plt.title(f"Fronteras SVM — {PC_X} vs {PC_Y} (viz)")
plt.tight_layout()
plt.savefig(OUTDIR / "fronteras_svm.png", dpi=200)
plt.show()
print("✅ fronteras_svm.png guardado.")

# === GUARDAR MODELO, COLUMNAS ESPERADAS, RESUMEN ===
import joblib

OUTDIR.mkdir(parents=True, exist_ok=True)

modelo_path = OUTDIR / "modelo_svm.pkl"
cols_path   = OUTDIR / "expected_columns.json"
resumen_path = OUTDIR / "svm_resumen.json"

joblib.dump(best, modelo_path)
print("✅ Modelo guardado en:", modelo_path)

with open(cols_path, "w", encoding="utf-8") as f:
    json.dump(
        {"columns": list(X_train.columns), "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")},
        f, ensure_ascii=False, indent=2
    )
print("✅ expected_columns.json guardado.")

scoring_str = getattr(scoring_cv, "__name__", str(scoring_cv))
resumen = {
    "best_params": grid_cv.best_params_,
    "cv_best_score": float(grid_cv.best_score_),
    "scoring_cv": scoring_str,
    "classes": [str(c) for c in classes_],
    "pos_label": str(pos_label) if K == 2 else None,
    "n_features": X_train.shape[1],
    "n_classes": K
}
with open(resumen_path, "w", encoding="utf-8") as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)
print("✅ svm_resumen.json guardado.")

print("\nArtefactos clave escritos en:", OUTDIR.resolve())

# === POLÍTICA DE INFERENCIA (NUEVO) ===
policy_path = OUTDIR / "inference_policy.json"
if K == 2 and alpha_opt is not None:
    inference_policy = {
        "task": "binary",
        "decision": {
            "type": "threshold",
            "alpha": float(alpha_opt),
            "criterion": "f1" if alpha_criterion == "f1" else "accuracy",
            "pos_label": str(pos_label)
        },
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }
else:
    inference_policy = {
        "task": "multiclass",
        "decision": {"type": "argmax_proba"},
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }
with open(policy_path, "w", encoding="utf-8") as f:
    json.dump(inference_policy, f, ensure_ascii=False, indent=2)
print("✅ inference_policy.json guardado.")

# === ZIP DE ARTEFACTOS ===
zip_path = OUTDIR / "svm_multiclase_artifacts_bundle.zip"

candidates = [
    "modelo_svm.pkl",
    "expected_columns.json",
    "svm_resumen.json",
    "classification_report.txt",
    "matriz_confusion.png",
    "fronteras_svm.png",
    "T_train_final_objetivo_scores.csv",
    "T_test_final_objetivo_scores.csv",
    "inference_policy.json"
]

present = [str(OUTDIR / f) for f in candidates if (OUTDIR / f).exists()]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in present:
        zf.write(f, arcname=os.path.basename(f))

print("✅ ZIP creado en:", zip_path)
print("Incluidos en el bundle:")
for f in present:
    print("  -", os.path.basename(f))
